In [1]:
import nltk
import math
import pandas as pd
from collections import Counter, defaultdict

# Download stopwords if not already downloaded
nltk.download('stopwords')



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dzwar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
import os
import nltk
import math
import pandas as pd
from collections import defaultdict, Counter

def make_desc(mode):
    document_count = defaultdict(set)  # Track unique documents for each token
    docs = []
    terms = []
    frequencies = []
    max_frequencies = []
    poids_values = []
    occurrences = []
    positions_list = []  # New list to store positions for each token
    
    term_frequency_data = []  # Temporary storage for term frequency data
    
    for x in range(6):
        doc_path = f'Collection/D{x+1}.txt'
        with open(doc_path, 'r') as document:
            text = document.read()
           
        text = text.lower()
        tempo_text = text.lower()
        stop_words = set(nltk.corpus.stopwords.words('english'))

        # Tokenization and preprocessing based on mode
        if mode == "Split":
            tokens = text.split()
        elif mode == "SplitPorter":
            tokens = text.split()
            tokens = [token for token in tokens if token.lower() not in stop_words]
            tokens = [nltk.PorterStemmer().stem(token) for token in tokens]
        elif mode == "SplitLancaster":
            tokens = text.split()
            tokens = [token for token in tokens if token.lower() not in stop_words]
            tokens = [nltk.LancasterStemmer().stem(token) for token in tokens]
        elif mode == "Token":
            tokenizer = nltk.RegexpTokenizer(r'(?:[A-Za-z]\.)+|[A-Za-z]+[\-@]\d+(?:\.\d+)?|\d+[A-Za-z]+|\d+(?:[\.\,\-]\d+)?%?|\w+(?:[\-/]\w+)*')
            tokens = tokenizer.tokenize(text)
        elif mode == "TokenPorter":
            tokenizer = nltk.RegexpTokenizer(r'(?:[A-Za-z]\.)+|[A-Za-z]+[\-@]\d+(?:\.\d+)?|\d+[A-Za-z]+|\d+(?:[\.\,\-]\d+)?%?|\w+(?:[\-/]\w+)*')
            tokens = tokenizer.tokenize(text)
            tokens = [token for token in tokens if token.lower() not in stop_words]
            tokens = [nltk.PorterStemmer().stem(token) for token in tokens]
        elif mode == "TokenLancaster":
            tokenizer = nltk.RegexpTokenizer(r'(?:[A-Za-z]\.)+|[A-Za-z]+[\-@]\d+(?:\.\d+)?|\d+[A-Za-z]+|\d+(?:[\.\,\-]\d+)?%?|\w+(?:[\-/]\w+)*')
            tokens = tokenizer.tokenize(text)
            tokens = [token for token in tokens if token.lower() not in stop_words]
            tokens = [nltk.LancasterStemmer().stem(token) for token in tokens]

        # Stopwords
        tokens = [token for token in tokens if token.lower() not in stop_words]
        
        # Get positions of tokens
        positions = specific_word_positions(tempo_text, tokens, mode)
        
        term_count = Counter(tokens)
        max_freq = max(term_count.values())
        
        unique_tokens = set(tokens)
        for token in unique_tokens:
            document_count[token].add(x + 1)  
        
        for token, freq in term_count.items():
            term_frequency_data.append((x + 1, token, freq, max_freq, positions.get(token, [])))

    
    for doc_id, token, freq, max_freq, position in term_frequency_data:
        
        docs.append(doc_id)
        terms.append(token)
        frequencies.append(freq)
        max_frequencies.append(max_freq)
        positions_list.append(position)  # Add position list to the main list
        
        occurrence = len(document_count[token])
        poids = (freq / max_freq) * math.log10((6 / occurrence) + 1)  
        
        poids_values.append(poids)
        occurrences.append(occurrence)

    df = pd.DataFrame({
        'Document': docs,
        'Token': terms,
        'Frequency': frequencies,
        'Max_Frequency': max_frequencies,
        'Occurrence': occurrences,
        'Poids': poids_values,
        'Position': positions_list  # Add Position column
    })

    # Sort the DataFrame by the 'Token' column alphabetically
    df = df.sort_values(by='Token').reset_index(drop=True)

    # Save the sorted DataFrame to a CSV file
    output_file_path = os.path.join('test', f'{mode}.csv')
    df.to_csv(output_file_path, index=False)
    print(f"Results saved to {output_file_path}")


def specific_word_positions(text, target_words, mode):
    word_pos = defaultdict(list)

    target_words_set = set(target_words)
    tokenizer = nltk.RegexpTokenizer(
        r'(?:[A-Za-z]\.)+|[A-Za-z]+[\-@]\d+(?:\.\d+)?|\d+[A-Za-z]+|\d+(?:[\.\,\-]\d+)?%?|\w+(?:[\-/]\w+)*')
    text = text.lower()
    
    if mode == "Split":
        tokens = text.split()
    elif mode == "SplitPorter":
        tokens = text.split()
        tokens = [nltk.PorterStemmer().stem(token) for token in tokens]
    elif mode == "SplitLancaster":
        tokens = text.split()
        tokens = [nltk.LancasterStemmer().stem(token) for token in tokens]
    elif mode == "Token":
        tokens = tokenizer.tokenize(text)
    elif mode == "TokenPorter":
        tokens = tokenizer.tokenize(text)
        tokens = [nltk.PorterStemmer().stem(token) for token in tokens]
    elif mode == "TokenLancaster":
        tokens = tokenizer.tokenize(text)
        tokens = [nltk.LancasterStemmer().stem(token) for token in tokens]

    tokens = [token.lower() for token in tokens]
    for index, word in enumerate(tokens, start=1):
        if word in target_words_set:  # Only track positions for target words
            word_pos[word].append(index)

    return dict(word_pos)  # Convert back to a regular dictionary


In [3]:
for mode in ["Split", "SplitPorter", "SplitLancaster", "Token", "TokenPorter", "TokenLancaster"]:
    make_desc(mode)

Results saved to test\Split.csv
Results saved to test\SplitPorter.csv
Results saved to test\SplitLancaster.csv
Results saved to test\Token.csv
Results saved to test\TokenPorter.csv
Results saved to test\TokenLancaster.csv
